### Blueheart Tools
- Notebook to go through cruise folders on blueheart and do:
    - check if cruises, grids coverages exist
    - create lists of (missing) products like bathy grids, zero grids and coverage geopackages
    - For single files: To generate missing products, use `Create_cov.sh` script
    - For multiple files: It might be more handy to batch process and use file lists. Proceed to 
    - **⚡ Work on local disk, not directly on blueheart when generating new products!**
    - Note that gdal sometimes is annoying to use - it may through error messages but still do the job, depending on gdal       version. Ignore the error messages if results are satifactory

- **⚡ Note that you need to be connected to Blueheart**
- **⚡ You need to have gdal installed, best via conda. Use the provided environment.yml if unsure.** 
- **⚡ Note that you need to change platform name and maybe sometimes paths to directories**


In [ ]:
import shutil
import geopandas as gpd
import pandas as pd
import os
import numpy as np
from pathlib import Path
import glob

In [ ]:
# Set vessel name for python
platform = "MERIAN"  # "MERIAN", "METEOR", "SONNE"

#### 0. Create product lists 
- Remove older versions
- Create lists for existing and missing files, useful also for finding erroneously named datasets etc.
- **⚡ Remember to update these if new products are generated**

In [ ]:
# Set working directory (equivalent of cd)
base_dir = f"/Volumes/bathymetry/_blueheart/00_{platform}/{platform}_GEOMAR"
os.chdir(base_dir)

# Set list names and paths

cruise_list = f'/Volumes/bathymetry/_blueheart/00_{platform}/{platform}_GEOMAR/{platform}_cruise_list.txt'
grid_list = f'/Volumes/bathymetry/_blueheart/00_{platform}/{platform}_GEOMAR/{platform}_grid_list.txt'
zero_list = f'/Volumes/bathymetry/_blueheart/00_{platform}/{platform}_GEOMAR/{platform}_zero_list.txt'
gpkg_list = f'/Volumes/bathymetry/_blueheart/00_{platform}/{platform}_GEOMAR/{platform}_gpkg_list.txt'
missing_grid_list = f'/Volumes/bathymetry/_blueheart/00_{platform}/{platform}_GEOMAR/{platform}_missing_grid_list.txt'
missing_zgrid_list = f'/Volumes/bathymetry/_blueheart/00_{platform}/{platform}_GEOMAR/{platform}_missing_zero_list.txt'
missing_gpkg_list = f'/Volumes/bathymetry/_blueheart/00_{platform}/{platform}_GEOMAR/{platform}_missing_gpkg_list.txt'

# Files to remove (equivalent of rm)
for fname in [f"{platform}_grid_list.txt", f"{platform}_cruise_list.txt",
              f"{platform}_zero_list.txt", f"{platform}_gpkg_list.txt",
              f"{platform}_missing_zero_list.txt", f"{platform}_missing_gpkg_list.txt",
              f"{platform}_missing_grid_list.txt"]:
    try:
        os.remove(fname)
    except FileNotFoundError:
        pass  

# 1. List */*/_grd/*EPSG3395.tif
grid_files = glob.glob("*/_grd/*EPSG3395.tif", recursive=True)
grid_files += glob.glob("*/*/_grd/*EPSG3395.tif", recursive=True)  # Just in case of two levels
grid_files = [os.path.abspath(f) for f in grid_files]
with open(grid_list, "w") as f:
    f.write("\n".join(sorted(grid_files)))

# 2. List directories in base dir (like `ls > SONNE_cruise_list.txt`)
entries = os.listdir(base_dir)
entries = [os.path.abspath(e) for e in entries]
with open(cruise_list, "w") as f:
    f.write("\n".join(sorted(entries)))

# 3. List */*/_grd/*_zero.tif
zero_files = glob.glob("*/_grd/*_zero.tif", recursive=True)
zero_files += glob.glob("*/*/_grd/*_zero.tif", recursive=True)
zero_files = [os.path.abspath(f) for f in zero_files]
with open(zero_list, "w") as f:
    f.write("\n".join(sorted(zero_files)))

# 5. List */*/_cov/*_Area.gpkg
gpkg_files = glob.glob("*/_cov/*_Coverage.gpkg")
gpkg_files += glob.glob("*/*/_cov/*_Coverage.gpkg")
gpkg_files = [os.path.abspath(f) for f in gpkg_files]
with open(gpkg_list, "w") as f:
    f.write("\n".join(sorted(gpkg_files)))


In [ ]:
# Find missing datasets by comparing the cruise_list (= all available cruises) with the first letters of the respective product (=cruise name). 
# If there is a cruise in the cruise_list which has no equivalent in one of the datasets' name, they will be marked as missing.
# Important note: If names are misspelled, these datasets will be marked as missing! 
# Most common mistake: Cruise leg names should ALWAYS be written like e.g. SO301-1, and NOT(!) SO301_1. 
# Currently, there is no check for such spelling mistakes hence if wrong, the respective datasets will be marked as missing and have to be corrected manually.

with open(cruise_list, 'r') as c_fi:
        cruise_path = c_fi.readlines()
        CRUISE = []
        for c_path in cruise_path:
            if not c_path.endswith('*Store'):
                cruise = c_path.split('/')[-1]
                cruise = cruise.strip('\n')
                CRUISE.append(cruise)

with open(grid_list, 'r') as g_fi:
    grid_path = g_fi.readlines()
    GRID = []
    GRID_PATH = []
    for g_path in grid_path:
        grid = g_path.split('/')[-4]
        GRID.append(grid)
        GRID_PATH.append(g_path)

with open(zero_list, 'r') as z_fi:
    zero_path = z_fi.readlines()
    ZGRID = []
    ZGRID_PATH = []
    for z_path in zero_path:
        zgrid = z_path.split('/')[-4]
        ZGRID.append(zgrid)
        ZGRID_PATH.append(z_path)
    

with open(gpkg_list, 'r') as gp_fi:
    gpkg_path = gp_fi.readlines()
    GPKG = []
    for gp_path in gpkg_path:
        gpkg = gp_path.split('/')[-1]
        gpkg = gpkg.split('_')[0]
        GPKG.append(gpkg)


# Compare cruise lists and grid list to evaluate where grids are missing
missing_grids = [ cr for cr in CRUISE if cr not in GRID and os.path.isdir(cr) and not cr.endswith('Cov') ]
print(f"missing_grid: {np.array(missing_grids)}")
np.savetxt(missing_grid_list, missing_grids, delimiter=';', fmt='%s')

# add grids paths where no zero grid exists to list
missing_zgrids = [ zgp for (zg, zgp) in zip(GRID, GRID_PATH) if zg not in ZGRID and os.path.isdir(zg) and not zg.endswith('Cov') ]
print(f"missing_zgrid: {np.array(missing_zgrids)}")
np.savetxt(missing_zgrid_list, missing_zgrids, delimiter=';', fmt='%s')

missing_gpkg = [ gpp for (gp, gpp) in zip(ZGRID, ZGRID_PATH) if gp not in GPKG and os.path.isdir(gp) and not gp.endswith('Cov') ]
print(f"missing_gpkg: {np.array(missing_gpkg)}")
np.savetxt(missing_gpkg_list, missing_gpkg, delimiter=';', fmt='%s')